In [1]:
try:
    import tensorflow as tf
    print("TensorFlow version:", tf.__version__)
    print("TensorFlow is ready!")
except ImportError:
    print("TensorFlow is not installed.")

TensorFlow is not installed.


In [2]:
%pip install tensorflow

   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/351.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/351.2 MB 669.0 kB/s eta 0:08:45
   ---------------------------------------- 0.5/351.2 MB 669.0 kB/s eta 0:08:45
   ---------------------------------------- 0.8/351.2 MB 707.7 kB/s eta 0:08:16
   ---------------------------------------- 1.0/351.2 MB 808.4 kB/s eta 0:07:14
   ---------------------------------------- 1.3/351.2 MB 838.7 kB/s eta 0:06:58
   ---------------------------------------- 1.3/351.2 MB 838.7 kB/s eta 0:06:58
   ---------------------------------------- 1.6/351.2 MB 846.4 kB/s eta 0:06:54
   ---------------------------------------- 1.8/351.2 MB 836.9 kB/s eta 0:06:58
   ---------------------------------------- 2.1/351.2 MB 871.1 kB/s eta 0:06

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [3]:
import tensorflow as tf
import google.protobuf

print("TensorFlow version:", tf.__version__)
print("Protobuf version:", google.protobuf.__version__)

print("TensorFlow is working successfully!")

TensorFlow version: 2.21.0
Protobuf version: 7.36.2
TensorFlow is working successfully!


In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf

# Load feature-engineered dataset
df = pd.read_csv(
    "../data/traffic_features.csv",
    parse_dates=["datetime"]
)

# Sort by grid and timestamp
df = df.sort_values(
    ["GridID", "datetime"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Unique grids:", df["GridID"].nunique())
print("Missing values:", df.isnull().sum().sum())

display(df.head())

Dataset shape: (20880, 19)
Unique grids: 5
Missing values: 0


,GridID,TimeInterval,internet,datetime,local_datetime,hour,date,lag_1,lag_2,lag_3,lag_6,lag_144,rolling_mean_6,rolling_std_6,rolling_mean_144,rolling_std_144,day_of_week,is_weekend,minute_slot
0,1,1383346800000,7.837850,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,13.228100,7.957519,9.272482,8.635192,11.028366,9.802627,1.900651,10.465614,3.320239,5,1,0
1,1,1383347400000,8.621555,2013-11-01 23:10:00+00:00,2013-11-02 00:10:00+01:00,0,2013-11-02,7.837850,13.228100,7.957519,10.659124,11.127101,9.669737,2.022566,10.443457,3.327096,5,1,1
2,1,1383348000000,7.825424,2013-11-01 23:20:00+00:00,2013-11-02 00:20:00+01:00,0,2013-11-02,8.621555,7.837850,13.228100,9.063345,10.892771,9.330142,1.994077,10.426058,3.330046,5,1,2
3,1,1383348600000,7.964082,2013-11-01 23:30:00+00:00,2013-11-02 00:30:00+01:00,0,2013-11-02,7.825424,8.621555,7.837850,9.272482,8.622424,9.123822,2.088985,10.404757,3.336843,5,1,3
4,1,1383349200000,6.500324,2013-11-01 23:40:00+00:00,2013-11-02 00:40:00+01:00,0,2013-11-02,7.964082,7.825424,8.621555,7.957519,8.009928,8.905755,2.138077,10.400185,3.339752,5,1,4


In [6]:
import numpy as np

# Get unique timestamps in chronological order
timestamps = np.sort(df["datetime"].unique())

# Calculate split boundaries
train_end = int(len(timestamps) * 0.70)
val_end = int(len(timestamps) * 0.85)

# Create timestamp sets
train_times = set(timestamps[:train_end])
val_times = set(timestamps[train_end:val_end])
test_times = set(timestamps[val_end:])

# Split dataset
train_df = df[df["datetime"].isin(train_times)].copy()
val_df = df[df["datetime"].isin(val_times)].copy()
test_df = df[df["datetime"].isin(test_times)].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Testing shape:", test_df.shape)

Training shape: (14615, 19)
Validation shape: (3130, 19)
Testing shape: (3135, 19)


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Build LSTM model
lstm_model = Sequential([
    Input(shape=(6, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

# Compile model
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

# Display model architecture
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 6, 64)               │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 6, 64)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 32)                  │          12,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 29,857 (116.63 KB)

 Trainable params: 29,857 (116.63 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Train LSTM
history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

print("LSTM training completed successfully!")

NameError: name 'X_train_seq' is not defined

In [9]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Load data if necessary
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv(
        "../data/traffic_features.csv",
        parse_dates=["datetime"]
    )

df = df.sort_values(["GridID", "datetime"]).reset_index(drop=True)

# Same chronological split as ML models
timestamps = np.sort(df["datetime"].unique())
train_end = int(len(timestamps) * 0.70)
val_end = int(len(timestamps) * 0.85)

train_times = set(timestamps[:train_end])
val_times = set(timestamps[train_end:val_end])
test_times = set(timestamps[val_end:])

train_df = df[df["datetime"].isin(train_times)]

# Fit scaler using training data only
scaler = MinMaxScaler()
scaler.fit(train_df[["internet"]])

df["internet_scaled"] = scaler.transform(df[["internet"]])

sequence_length = 6

X_train_seq, y_train_seq = [], []
X_val_seq, y_val_seq = [], []
X_test_seq, y_test_seq = [], []

for grid_id, group in df.groupby("GridID"):

    group = group.sort_values("datetime")
    values = group["internet_scaled"].to_numpy()
    times = group["datetime"].to_numpy()

    for i in range(sequence_length, len(group)):
        X = values[i-sequence_length:i]
        y = values[i]
        target_time = times[i]

        if target_time in train_times:
            X_train_seq.append(X)
            y_train_seq.append(y)
        elif target_time in val_times:
            X_val_seq.append(X)
            y_val_seq.append(y)
        elif target_time in test_times:
            X_test_seq.append(X)
            y_test_seq.append(y)

X_train_seq = np.array(X_train_seq).reshape(-1, 6, 1)
X_val_seq = np.array(X_val_seq).reshape(-1, 6, 1)
X_test_seq = np.array(X_test_seq).reshape(-1, 6, 1)

y_train_seq = np.array(y_train_seq)
y_val_seq = np.array(y_val_seq)
y_test_seq = np.array(y_test_seq)

print("Training:", X_train_seq.shape)
print("Validation:", X_val_seq.shape)
print("Testing:", X_test_seq.shape)

Training: (14585, 6, 1)
Validation: (3130, 6, 1)
Testing: (3135, 6, 1)


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict normalized traffic
lstm_pred_scaled = lstm_model.predict(X_val_seq)

# Convert predictions back to original traffic scale
lstm_pred = scaler.inverse_transform(
    lstm_pred_scaled
).flatten()

# Convert actual values back to original scale
lstm_actual = scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

# Calculate evaluation metrics
lstm_mae = mean_absolute_error(lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
lstm_r2 = r2_score(lstm_actual, lstm_pred)

print("=== LSTM Model (Validation) ===")
print(f"MAE  : {lstm_mae:.4f}")
print(f"RMSE : {lstm_rmse:.4f}")
print(f"R²   : {lstm_r2:.4f}")

98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
=== LSTM Model (Validation) ===
MAE  : 14.6641
RMSE : 18.1030
R²   : -1.6085


In [11]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("LSTM Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error (Scaled)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Epochs trained:", len(history.history["loss"]))
print("Best validation loss:", min(history.history["val_loss"]))

NameError: name 'history' is not defined

<Figure size 1000x500 with 0 Axes>

In [12]:
print("LSTM model available:", "lstm_model" in globals())
print("Training history available:", "history" in globals())
print("Training sequences available:", "X_train_seq" in globals())

if "lstm_model" in globals():
    print("Model optimizer:", lstm_model.optimizer)

LSTM model available: True
Training history available: False
Training sequences available: True
Model optimizer: <keras.src.optimizers.adam.Adam object at 0x0000022841574AD0>


In [13]:
import pandas as pd
import numpy as np

# Generate predictions
pred_scaled = lstm_model.predict(X_val_seq, verbose=0)

# Convert to original activity scale
predictions = scaler.inverse_transform(pred_scaled).flatten()

actual = scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

# Compare actual and predicted traffic
comparison = pd.DataFrame({
    "Actual": actual,
    "Predicted": predictions,
    "Error": actual - predictions
})

print("Prediction statistics:")
print(comparison.describe().round(3))

print("\nFirst 10 predictions:")
display(comparison.head(10))

Prediction statistics:
         Actual  Predicted     Error
count  3130.000   3130.000  3130.000
mean     18.533      3.869    14.664
std      11.210      0.648    10.617
min       3.710      2.996     0.708
25%      10.694      3.416     7.232
50%      16.262      3.751    12.447
75%      23.407      4.186    19.285
max      83.716      7.497    76.442

First 10 predictions:


,Actual,Predicted,Error
0,8.308100,3.241870,5.066230
1,9.267362,3.275731,5.991631
2,12.159556,3.304481,8.855075
3,11.002251,3.340001,7.662250
4,10.386217,3.376298,7.009919
5,10.270207,3.382074,6.888133
6,11.833324,3.370235,8.463089
7,9.478777,3.398974,6.079803
8,11.092258,3.404006,7.688252
9,10.031691,3.390624,6.641067


In [14]:
# Check training data and scaling
print("Original training traffic statistics:")
print(train_df["internet"].describe())

print("\nScaler minimum:", scaler.data_min_)
print("Scaler maximum:", scaler.data_max_)

print("\nFirst training sequence (scaled):")
print(X_train_seq[0].flatten())

print("\nSame sequence (original scale):")
print(
    scaler.inverse_transform(
        X_train_seq[0].reshape(-1, 1)
    ).flatten()
)

print("\nFirst training target (original scale):")
print(
    scaler.inverse_transform(
        y_train_seq[:1].reshape(-1, 1)
    ).flatten()
)

Original training traffic statistics:
count    14615.000000
mean        17.463795
std         10.477874
min          2.906193
25%          9.885679
50%         15.030232
75%         22.683799
max         85.071380
Name: internet, dtype: float64

Scaler minimum: [2.9061933]
Scaler maximum: [85.07138]

First training sequence (scaled):
[0.06002124 0.06955941 0.05987001 0.06155756 0.04374274 0.05907556]

Same sequence (original scale):
[7.83785   8.621555  7.825424  7.964082  6.500324  7.7601476]

First training target (original scale):
[7.8945804]


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict training data
train_pred_scaled = lstm_model.predict(
    X_train_seq,
    verbose=0
)

# Convert predictions to original scale
train_pred = scaler.inverse_transform(
    train_pred_scaled
).flatten()

train_actual = scaler.inverse_transform(
    y_train_seq.reshape(-1, 1)
).flatten()

# Calculate evaluation metrics
print("=== LSTM Training Performance ===")

print("MAE:", round(
    mean_absolute_error(train_actual, train_pred), 4
))

print("RMSE:", round(
    np.sqrt(mean_squared_error(train_actual, train_pred)), 4
))

print("R²:", round(
    r2_score(train_actual, train_pred), 4
))

print("\nActual training mean:", round(train_actual.mean(), 4))
print("Predicted training mean:", round(train_pred.mean(), 4))

=== LSTM Training Performance ===
MAE: 13.6708
RMSE: 16.9053
R²: -1.6012

Actual training mean: 17.478
Predicted training mean: 3.8072


In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Initialize a fresh LSTM model
lstm_model = Sequential([
    Input(shape=(6, 1)),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1)
])

lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

# Train and SAVE the training history
history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30,
    batch_size=32,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ],
    verbose=1
)

print("\nTraining completed!")
print("Epochs trained:", len(history.history["loss"]))
print("Best validation loss:", min(history.history["val_loss"]))

Epoch 1/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - loss: 0.0043 - mae: 0.0397 - val_loss: 0.0026 - val_mae: 0.0333
Epoch 2/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0028 - mae: 0.0327 - val_loss: 0.0024 - val_mae: 0.0309
Epoch 3/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0028 - mae: 0.0320 - val_loss: 0.0024 - val_mae: 0.0304
Epoch 4/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0027 - mae: 0.0314 - val_loss: 0.0024 - val_mae: 0.0306
Epoch 5/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0027 - mae: 0.0313 - val_loss: 0.0024 - val_mae: 0.0317
Epoch 6/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0313 - val_loss: 0.0024 - val_mae: 0.0325
Epoch 7/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 0.0026 - mae: 0.0311 - val_loss: 0.0025 - val_mae: 0.0324
Epoch 8/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0310 - val_loss: 0.0023 - val_mae: 0.0300
Epoch 9/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - lo

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict validation data
lstm_pred_scaled = lstm_model.predict(
    X_val_seq,
    verbose=0
)

# Convert predictions to original scale
lstm_pred = scaler.inverse_transform(
    lstm_pred_scaled
).flatten()

# Convert actual values to original scale
lstm_actual = scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

# Calculate evaluation metrics
lstm_mae = mean_absolute_error(lstm_actual, lstm_pred)
lstm_rmse = np.sqrt(mean_squared_error(lstm_actual, lstm_pred))
lstm_r2 = r2_score(lstm_actual, lstm_pred)

print("=== Retrained LSTM (Validation) ===")
print(f"MAE  : {lstm_mae:.4f}")
print(f"RMSE : {lstm_rmse:.4f}")
print(f"R²   : {lstm_r2:.4f}")

=== Retrained LSTM (Validation) ===
MAE  : 2.4712
RMSE : 3.9250
R²   : 0.8774


In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dense
from tensorflow.keras.optimizers import Adam

# Build GRU model
gru_model = Sequential([
    Input(shape=(6, 1)),
    GRU(32),
    Dense(16, activation="relu"),
    Dense(1)
])

# Compile GRU model
gru_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

# Display model architecture
gru_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ gru (GRU)                            │ (None, 32)                  │           3,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              17 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,905 (15.25 KB)

 Trainable params: 3,905 (15.25 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict validation traffic
gru_pred_scaled = gru_model.predict(
    X_val_seq,
    verbose=0
)

# Convert predictions to original traffic scale
gru_pred = scaler.inverse_transform(
    gru_pred_scaled
).flatten()

# Convert actual values to original scale
gru_actual = scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

# Calculate evaluation metrics
gru_mae = mean_absolute_error(gru_actual, gru_pred)
gru_rmse = np.sqrt(mean_squared_error(gru_actual, gru_pred))
gru_r2 = r2_score(gru_actual, gru_pred)

print("=== GRU Model (Validation) ===")
print(f"MAE  : {gru_mae:.4f}")
print(f"RMSE : {gru_rmse:.4f}")
print(f"R²   : {gru_r2:.4f}")

=== GRU Model (Validation) ===
MAE  : 14.1553
RMSE : 17.4980
R²   : -1.4371


In [20]:
import pandas as pd

# Compare actual and predicted values
gru_comparison = pd.DataFrame({
    "Actual": gru_actual,
    "Predicted": gru_pred,
    "Error": gru_actual - gru_pred
})

print("=== GRU Prediction Statistics ===")
print(gru_comparison.describe().round(3))

print("\nFirst 10 predictions:")
display(gru_comparison.head(10))

print("\nGRU training epochs:",
      len(gru_history.history["loss"]))

print("Best validation loss:",
      min(gru_history.history["val_loss"]))

=== GRU Prediction Statistics ===
         Actual  Predicted     Error
count  3130.000   3130.000  3130.000
mean     18.533      4.378    14.155
std      11.210      0.996    10.288
min       3.710      3.024     0.642
25%      10.694      3.676     6.965
50%      16.262      4.203    12.027
75%      23.407      4.859    18.574
max      83.716      9.951    73.978

First 10 predictions:


,Actual,Predicted,Error
0,8.308100,3.612924,4.695176
1,9.267362,3.517519,5.749843
2,12.159556,3.522299,8.637257
3,11.002251,3.656546,7.345705
4,10.386217,3.667595,6.718622
5,10.270207,3.643895,6.626312
6,11.833324,3.625684,8.207640
7,9.478777,3.694630,5.784147
8,11.092258,3.617238,7.475019
9,10.031691,3.651885,6.379806


NameError: name 'gru_history' is not defined

In [21]:
import pandas as pd

gru_comparison = pd.DataFrame({
    "Actual": gru_actual,
    "Predicted": gru_pred,
    "Error": gru_actual - gru_pred
})

print("=== GRU Prediction Statistics ===")
print(gru_comparison.describe().round(3))

print("\nFirst 10 predictions:")
display(gru_comparison.head(10))

=== GRU Prediction Statistics ===
         Actual  Predicted     Error
count  3130.000   3130.000  3130.000
mean     18.533      4.378    14.155
std      11.210      0.996    10.288
min       3.710      3.024     0.642
25%      10.694      3.676     6.965
50%      16.262      4.203    12.027
75%      23.407      4.859    18.574
max      83.716      9.951    73.978

First 10 predictions:


,Actual,Predicted,Error
0,8.308100,3.612924,4.695176
1,9.267362,3.517519,5.749843
2,12.159556,3.522299,8.637257
3,11.002251,3.656546,7.345705
4,10.386217,3.667595,6.718622
5,10.270207,3.643895,6.626312
6,11.833324,3.625684,8.207640
7,9.478777,3.694630,5.784147
8,11.092258,3.617238,7.475019
9,10.031691,3.651885,6.379806


In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Create a fresh GRU model
gru_model = Sequential([
    Input(shape=(6, 1)),
    GRU(32),
    Dense(16, activation="relu"),
    Dense(1)
])

# Compile the model
gru_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

# Train and store history
gru_history = gru_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=30,
    batch_size=32,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ],
    verbose=1
)

print("\nGRU training completed!")
print("Epochs trained:", len(gru_history.history["loss"]))
print("Best validation loss:", min(gru_history.history["val_loss"]))

Epoch 1/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.0047 - mae: 0.0411 - val_loss: 0.0024 - val_mae: 0.0304
Epoch 2/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0027 - mae: 0.0315 - val_loss: 0.0023 - val_mae: 0.0304
Epoch 3/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0027 - mae: 0.0312 - val_loss: 0.0024 - val_mae: 0.0332
Epoch 4/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0310 - val_loss: 0.0024 - val_mae: 0.0317
Epoch 5/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0311 - val_loss: 0.0024 - val_mae: 0.0332
Epoch 6/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0310 - val_loss: 0.0024 - val_mae: 0.0307
Epoch 7/30
456/456 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.0026 - mae: 0.0309 - val_loss: 0.0024 - val_mae: 0.0309

GRU training completed!
Epochs trained: 7
Best validation loss: 0.0023417160846292973


In [23]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Predict validation data
gru_pred_scaled = gru_model.predict(
    X_val_seq,
    verbose=0
)

# Convert predictions to original scale
gru_pred = scaler.inverse_transform(
    gru_pred_scaled
).flatten()

# Convert actual traffic to original scale
gru_actual = scaler.inverse_transform(
    y_val_seq.reshape(-1, 1)
).flatten()

# Calculate metrics
gru_mae = mean_absolute_error(gru_actual, gru_pred)
gru_rmse = np.sqrt(mean_squared_error(gru_actual, gru_pred))
gru_r2 = r2_score(gru_actual, gru_pred)

print("=== Retrained GRU (Validation) ===")
print(f"MAE  : {gru_mae:.4f}")
print(f"RMSE : {gru_rmse:.4f}")
print(f"R²   : {gru_r2:.4f}")

=== Retrained GRU (Validation) ===
MAE  : 2.5013
RMSE : 3.9761
R²   : 0.8742


In [24]:
import os

# Ensure models folder exists
os.makedirs("../models", exist_ok=True)

# Save trained GRU model
gru_model.save("../models/gru_model.keras")

print("GRU model saved successfully!")

GRU model saved successfully!


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np
import joblib

# Load previously saved models
lr_model = joblib.load("../models/linear_regression.pkl")
rf_model = joblib.load("../models/random_forest.pkl")
xgb_model = joblib.load("../models/xgboost.pkl")

# Select input features
features = [
    "GridID", "hour", "day_of_week", "is_weekend",
    "minute_slot", "lag_1", "lag_2", "lag_3",
    "lag_6", "lag_144", "rolling_mean_6",
    "rolling_std_6", "rolling_mean_144",
    "rolling_std_144"
]

# Prepare test data
X_test = test_df[features]
y_test = test_df["internet"]

# Models to evaluate
models = {
    "Persistence": None,
    "Historical Average": None,
    "Linear Regression": lr_model,
    "Random Forest": rf_model,
    "XGBoost": xgb_model
}

results = []

for name, model in models.items():

    if name == "Persistence":
        predictions = test_df["lag_1"]

    elif name == "Historical Average":
        predictions = test_df["rolling_mean_6"]

    else:
        predictions = model.predict(X_test)

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)

print("=== FINAL TEST RESULTS ===")
display(results_df.round(4))

=== FINAL TEST RESULTS ===


,Model,MAE,RMSE,R2
0,Persistence,2.7589,4.9404,0.7978
1,Historical Average,2.8919,4.9535,0.7967
2,Linear Regression,2.4840,4.3500,0.8432
3,Random Forest,2.4501,4.3093,0.8461
4,XGBoost,2.4744,4.3368,0.8442


In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Convert actual test values to original scale
y_test_actual = scaler.inverse_transform(
    y_test_seq.reshape(-1, 1)
).flatten()

results = []

for name, model in [
    ("LSTM", lstm_model),
    ("GRU", gru_model)
]:
    # Predict test traffic
    pred_scaled = model.predict(X_test_seq, verbose=0)

    # Convert predictions back to original scale
    predictions = scaler.inverse_transform(
        pred_scaled
    ).flatten()

    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test_actual, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test_actual, predictions)),
        "R2": r2_score(y_test_actual, predictions)
    })

deep_results = pd.DataFrame(results)

print("=== FINAL TEST RESULTS: DEEP LEARNING ===")
display(deep_results.round(4))

=== FINAL TEST RESULTS: DEEP LEARNING ===


,Model,MAE,RMSE,R2
0,LSTM,2.5196,4.3840,0.8408
1,GRU,2.5704,4.4622,0.8350


In [27]:
# Save LSTM and GRU test results
deep_results.to_csv(
    "../outputs/deep_learning_test_results.csv",
    index=False
)

print("Deep learning test results saved successfully!")
display(deep_results)

Deep learning test results saved successfully!


,Model,MAE,RMSE,R2
0,LSTM,2.519562,4.383991,0.840750
1,GRU,2.570417,4.462180,0.835019
